In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

print("Working directory:", ROOT)
print("Metadata exists:", (ROOT / "meta" / "esc50.csv").exists())
print("Audio folder exists:", (ROOT / "audio").is_dir())

df = pd.read_csv(ROOT / "meta" / "esc50.csv")

print("Dataset shape:", df.shape)
print("Unique classes:", df["target"].nunique())
print("Folds:", sorted(df["fold"].unique()))

Working directory: C:\Users\Suprakash Ghosh\Desktop\ESC-50
Metadata exists: True
Audio folder exists: True
Dataset shape: (2000, 7)
Unique classes: 50
Folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [2]:
from pathlib import Path
audio_dir = ROOT / "audio"
actual_files = {
    path.name for path in audio_dir.iterdir()
    if path.is_file()
}

metadata_files = set(df["filename"])
missing_files = metadata_files - actual_files
extra_files = actual_files - metadata_files
invalid_targets = df.loc[~df["target"].between(0, 49)]
invalid_folds = df.loc[~df["fold"].between(1, 5)]

class_counts = df.groupby("target").size()
print("Missing audio files:", len(missing_files))
print("Extra audio files:", len(extra_files))
print("Invalid target rows:", len(invalid_targets))
print("Invalid fold rows:", len(invalid_folds))
print("Classes with counts other than 40:",
      int((class_counts != 40).sum()))

print("\nMissing filenames (first 10):", sorted(missing_files)[:10])
print("Extra filenames (first 10):", sorted(extra_files)[:10])

Missing audio files: 0
Extra audio files: 0
Invalid target rows: 0
Invalid fold rows: 0
Classes with counts other than 40: 0

Missing filenames (first 10): []
Extra filenames (first 10): []


In [3]:
from pathlib import Path

test_file = ROOT / "tests" / "test_metadata_validation.py"

test_code = '''import pandas as pd

ROOT = __import__("pathlib").Path(__file__).resolve().parents[1]
META_PATH = ROOT / "meta" / "esc50.csv"
AUDIO_DIR = ROOT / "audio"


def test_metadata_has_expected_columns():
    metadata = pd.read_csv(META_PATH)
    expected = {
        "filename", "fold", "target", "category",
        "esc10", "src_file", "take"
    }
    assert expected.issubset(metadata.columns)


def test_metadata_has_no_missing_values_or_duplicate_filenames():
    metadata = pd.read_csv(META_PATH)
    assert not metadata.isnull().any().any()
    assert not metadata["filename"].duplicated().any()


def test_metadata_targets_are_valid():
    metadata = pd.read_csv(META_PATH)
    assert metadata["target"].between(0, 49).all()


def test_metadata_folds_are_valid():
    metadata = pd.read_csv(META_PATH)
    assert metadata["fold"].isin([1, 2, 3, 4, 5]).all()


def test_each_class_has_40_recordings():
    metadata = pd.read_csv(META_PATH)
    counts = metadata.groupby("target").size()
    assert len(counts) == 50
    assert (counts == 40).all()


def test_metadata_audio_files_exist():
    metadata = pd.read_csv(META_PATH)
    expected_files = set(metadata["filename"])
    actual_files = {path.name for path in AUDIO_DIR.iterdir() if path.is_file()}
    assert expected_files == actual_files
'''

if test_file.exists():
    raise FileExistsError(f"{test_file} already exists; inspect it before editing.")

test_file.write_text(test_code, encoding="utf-8")
print("Created:", test_file)

Created: C:\Users\Suprakash Ghosh\Desktop\ESC-50\tests\test_metadata_validation.py


In [4]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_metadata_validation.py"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

============================= test session starts =============================
platform win32 -- Python 3.10.20, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\Suprakash Ghosh\Desktop\ESC-50
configfile: pytest.ini
plugins: anyio-4.13.0, asyncio-1.4.0, cov-7.1.0
asyncio: mode=strict, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 6 items

tests\test_metadata_validation.py ......                                 [100%]

============================== warnings summary ===============================
..\..\..\..\Anaconda\envs\tfclean\lib\site-packages\_pytest\config\__init__.py:1464
  C:\Anaconda\envs\tfclean\lib\site-packages\_pytest\config\__init__.py:1464: PytestConfigWarning: Unknown config option: pep8maxlinelength
  
    self._warn_or_fail_if_strict(f"Unknown config option: {key}\n")

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================== 6 passed, 1 warning in 2.54s ====================

In [5]:
import subprocess

for command in [
    ["git", "status", "--short"],
    ["git", "diff", "--check"],
    ["git", "diff", "--", "tests/"],
]:
    result = subprocess.run(
        command, cwd=ROOT, capture_output=True, text=True
    )
    print(f"\n$ {' '.join(command)}")
    print(result.stdout or "(no output)")
    if result.stderr:
        print(result.stderr)


$ git status --short
?? Untitled.ipynb
?? tests/test_metadata_validation.py


$ git diff --check
(no output)

$ git diff -- tests/
(no output)


In [6]:
print(test_file.read_text(encoding="utf-8"))

import pandas as pd

ROOT = __import__("pathlib").Path(__file__).resolve().parents[1]
META_PATH = ROOT / "meta" / "esc50.csv"
AUDIO_DIR = ROOT / "audio"


def test_metadata_has_expected_columns():
    metadata = pd.read_csv(META_PATH)
    expected = {
        "filename", "fold", "target", "category",
        "esc10", "src_file", "take"
    }
    assert expected.issubset(metadata.columns)


def test_metadata_has_no_missing_values_or_duplicate_filenames():
    metadata = pd.read_csv(META_PATH)
    assert not metadata.isnull().any().any()
    assert not metadata["filename"].duplicated().any()


def test_metadata_targets_are_valid():
    metadata = pd.read_csv(META_PATH)
    assert metadata["target"].between(0, 49).all()


def test_metadata_folds_are_valid():
    metadata = pd.read_csv(META_PATH)
    assert metadata["fold"].isin([1, 2, 3, 4, 5]).all()


def test_each_class_has_40_recordings():
    metadata = pd.read_csv(META_PATH)
    counts = metadata.groupby("target").size()
    assert l

In [7]:
import subprocess

subprocess.run(
    ["git", "add", "tests/test_metadata_validation.py"],
    cwd=ROOT,
    check=True,
)

subprocess.run(
    ["git", "status", "--short"],
    cwd=ROOT,
    check=True,
)

CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [8]:
import subprocess

commands = [
    ["git", "status", "--short"],
    ["git", "diff", "--cached", "--check"],
    ["git", "diff", "--cached", "--stat"],
    ["git", "diff", "--cached", "--", "tests/test_metadata_validation.py"],
]

for command in commands:
    result = subprocess.run(
        command,
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"\n$ {' '.join(command)}")
    print(result.stdout or "(no output)")


$ git status --short
A  tests/test_metadata_validation.py
?? meta.ipynb


$ git diff --cached --check
(no output)

$ git diff --cached --stat
 tests/test_metadata_validation.py | 44 +++++++++++++++++++++++++++++++++++++++
 1 file changed, 44 insertions(+)


$ git diff --cached -- tests/test_metadata_validation.py
diff --git a/tests/test_metadata_validation.py b/tests/test_metadata_validation.py
new file mode 100644
index 0000000..6c7e8f9
--- /dev/null
+++ b/tests/test_metadata_validation.py
@@ -0,0 +1,44 @@
+import pandas as pd
+
+ROOT = __import__("pathlib").Path(__file__).resolve().parents[1]
+META_PATH = ROOT / "meta" / "esc50.csv"
+AUDIO_DIR = ROOT / "audio"
+
+
+def test_metadata_has_expected_columns():
+    metadata = pd.read_csv(META_PATH)
+    expected = {
+        "filename", "fold", "target", "category",
+        "esc10", "src_file", "take"
+    }
+    assert expected.issubset(metadata.columns)
+
+
+def test_metadata_has_no_missing_values_or_duplicate_filenames():
+    metad

In [9]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_metadata_validation.py"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Tests failed; do not commit yet."

============================= test session starts =============================
platform win32 -- Python 3.10.20, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\Suprakash Ghosh\Desktop\ESC-50
configfile: pytest.ini
plugins: anyio-4.13.0, asyncio-1.4.0, cov-7.1.0
asyncio: mode=strict, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 6 items

tests\test_metadata_validation.py ......                                 [100%]

============================== warnings summary ===============================
..\..\..\..\Anaconda\envs\tfclean\lib\site-packages\_pytest\config\__init__.py:1464
  C:\Anaconda\envs\tfclean\lib\site-packages\_pytest\config\__init__.py:1464: PytestConfigWarning: Unknown config option: pep8maxlinelength
  
    self._warn_or_fail_if_strict(f"Unknown config option: {key}\n")

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================== 6 passed, 1 warning in 1.31s ====================

In [10]:
subprocess.run(
    ["git", "commit", "-m", "Add ESC-50 metadata validation tests"],
    cwd=ROOT,
    check=True,
)

CompletedProcess(args=['git', 'commit', '-m', 'Add ESC-50 metadata validation tests'], returncode=0)

In [11]:
subprocess.run(["git", "status", "--short"], cwd=ROOT, check=True)
subprocess.run(["git", "log", "-1", "--oneline"], cwd=ROOT, check=True)

CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [13]:
assert df["target"].between(0, 49).all()
print("All class IDs are valid!")

All class IDs are valid!


In [14]:
metadata = pd.read_csv(ROOT / "meta" / "esc50.csv")
assert metadata["target"].between(0, 49).all()
print("All class IDs are valid!")

All class IDs are valid!


In [15]:
import subprocess

for command in [
    ["git", "status", "--short"],
    ["git", "log", "-1", "--oneline"],
]:
    result = subprocess.run(
        command, cwd=ROOT, capture_output=True, text=True
    )
    print("$", " ".join(command))
    print(result.stdout)

$ git status --short
?? meta.ipynb

$ git log -1 --oneline
c209e85 Add ESC-50 metadata validation tests

